# Lab 3 — European Rain Forecast: Score Your Model Locally

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-model.ipynb)

A forecast submitted to challenge 177 is scored 48 hours later. This
notebook replays the challenge on the past instead: every forecast
issued from 2025-01-01 to 2026-03-01, as the exact request the
challenge sends, scored the way the challenge scores it.

It scores two agents, the city × month climatology of the agent
notebook and a random forest, then leaves the model to you.

---

## 1. Setup and download

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and scikit-learn; the only install
is the ML-Arena client.

In [16]:
!pip install -q mlarena-sdk

zsh:1: command not found: pip


In [17]:
import os

MLARENA_API_KEY = "<your_api_key>"

import mlarena
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor

API_KEY = "mlk_user_be8edd1568655ff0_bbfbb4ff5c006ef6b5f81d63b919a32c"
CHALLENGE_ID = 177     

In [18]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

weather_europe_2020_2026.csv.gz 28.2 MB


---

## 2. The data, as the challenge sees it

The challenge sends the cities in its own fixed order, not the
file's alphabetical one. `X` is the whole file as an array (hours,
cities, features) in that order, so a request is a slice of it.

In [19]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

(54096, 45, 8) 2020-01-01 00:00:00+00:00 -> 2026-03-03 23:00:00+00:00


Models learn from **2020–2024**. The replay covers the forecasts
issued from **2025-01-01**, every 6 hours at the UTC hours the
challenge uses, up to the last one whose +48 h is in the file.
`issue_times(start, end)` keeps the issue hours whose +48 h falls
before `end`, so no training target reaches into the replay.

In [20]:
HORIZONS = [6, 48]
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours
SPLIT = hours.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))


def issue_times(start, end):
    """Issue hours in [start, end) with 48 h of history and their +48 h."""
    i = np.arange(max(start, 47), end - max(HORIZONS))
    return i[np.isin(hours.hour[i], ISSUE_HOURS)]


TRAIN = issue_times(0, SPLIT)
TEST = issue_times(SPLIT, len(hours))
print(len(TRAIN), "training issue times |", len(TEST), "replayed,",
      hours[TEST[0]], "->", hours[TEST[-1]])

7293 training issue times | 1700 replayed, 2025-01-01 05:00:00+00:00 -> 2026-03-01 23:00:00+00:00


---

## 3. The request and the local score

`make_request(i)` builds what the challenge sends when hour `i` is
the issue time (the keys are in the agent notebook, section 1): the
48 hours ending at `i`, the 45 cities in panel order, the 8 features.
The feed never reports `visibility`, and the challenge sends it as 0.
`check` validates a response the way the challenge does, and `crps`
is the score of the analysis notebook, section 9.

In [21]:
import json


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def window(i):
    """The history sent at issue hour i: (cities, 48 h, features)."""
    return np.nan_to_num(X[i - 47:i + 1].transpose(1, 0, 2), nan=0.0)


def make_request(i):
    return {
        "issue_time": iso(hours[i]),
        "timestamps": [iso(t) for t in hours[i - 47:i + 1]],
        "observed": [True] * 48,
        "cities": [{"name": n, "country": c,
                    "lat": float(coords.loc[j, "latitude"]),
                    "lon": float(coords.loc[j, "longitude"])}
                   for j, (n, c) in enumerate(PANEL)],
        "feature_names": FEATURES,
        "history": window(i).tolist(),
        "horizons": HORIZONS,
        "max_members": 100,
    }


def check(response, request):
    """The response as a (cities, horizons, M) array, or raise."""
    json.dumps(response, allow_nan=False)      # JSON, no NaN/inf
    ens = np.asarray(response["rain"], dtype=float)
    shape = (len(request["cities"]), len(request["horizons"]))
    assert ens.ndim == 3 and ens.shape[:2] == shape, ens.shape
    assert 1 <= ens.shape[2] <= request["max_members"], ens.shape
    assert np.isfinite(ens).all()
    return ens


def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # clipped at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread

`evaluate(agent, issues)` replays the challenge: for each issue time,
build the request, call `agent.predict`, and score both horizons
against the rain that fell. One row per run; the score is the mean
of the two CRPS, in mm, lower is better.

Consecutive runs share the weather, and the weather moves every
agent's CRPS together. `compare(a, b)` therefore compares two agents
run by run, and takes its standard error from weekly blocks of 28
runs.

In [22]:
def evaluate(agent, issues=TEST):
    """One row per run: CRPS per horizon (mean over cities), score."""
    rows = []
    for i in issues:
        request = make_request(i)
        ens = check(agent.predict(request), request)
        rows.append({f"CRPS {h}h": crps(ens[:, k], rain[i + h]).mean()
                     for k, h in enumerate(HORIZONS)})
    runs = pd.DataFrame(rows, index=hours[issues])
    runs["score"] = runs.mean(axis=1)
    return runs


def compare(a, b, week=28):
    """Mean of a - b over the same runs, and its weekly-block SE."""
    d = (a["score"] - b["score"]).to_numpy()
    blocks = d[:len(d) // week * week].reshape(-1, week).mean(axis=1)
    return d.mean(), blocks.std(ddof=1) / np.sqrt(len(blocks))

---

## 4. The baseline: city × month climatology

The agent notebook's model, refitted on 2020–2024 only: a table
that has seen the replayed months would score itself on its own
data.

In [23]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M


class Climatology:
    """For each city and valid month, M quantiles of the past rain."""

    def __init__(self, end):
        r, m = rain[:end], month[:end]
        self.table = np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                               for k in range(1, 13)])  # (12, cities, M)

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        months = [(issue + pd.Timedelta(hours=h)).month
                  for h in request["horizons"]]
        members = self.table[np.array(months) - 1]    # (2, cities, M)
        return {"rain": members.transpose(1, 0, 2).tolist()}


runs_clim = evaluate(Climatology(SPLIT))
print(runs_clim.mean().round(4).to_string())

CRPS 6h     0.0760
CRPS 48h    0.0758
score       0.0759


---

## 5. A random forest

Five features per city, read from the request: the rain now and
over the last 6 hours, the humidity and the cloud cover now, and the
month. The same `features` function builds the training rows from
`window(i)` and reads the live request, so the model sees at
prediction exactly what it saw in training.

In [27]:
RAIN, HUMIDITY, CLOUDS = (FEATURES.index(f)
                          for f in ("rain", "humidity", "clouds"))


def features(history, issue):
    """(cities, 5) features from a history (cities, 48 h, features)."""
    r = history[:, :, RAIN]
    return np.column_stack([
        r[:, -1],                          # rain now
        r[:, -6:].sum(axis=1),             # rain over the last 6 h
        history[:, -1, HUMIDITY],
        history[:, -1, CLOUDS],
        np.full(len(history), issue.month),
    ])


F = np.concatenate([features(window(i), hours[i]) for i in TRAIN])
print(F.shape, "training rows: issue times x cities")

(328185, 5) training rows: issue times x cities


One forest per horizon. A forest is an ensemble already: each of its
30 trees gives one member.

In [25]:
forests = {
    h: RandomForestRegressor(n_estimators=30, min_samples_leaf=100,
                             n_jobs=-1, random_state=0)
    .fit(F, np.concatenate([rain[i + h] for i in TRAIN]))
    for h in HORIZONS
}


class Forest:
    def __init__(self, forests):
        self.forests = forests

    def predict(self, request):
        F = features(np.asarray(request["history"]),
                     pd.Timestamp(request["issue_time"]))
        members = [np.stack([tree.predict(F)
                             for tree in self.forests[h].estimators_],
                            axis=-1)
                   for h in request["horizons"]]    # 2 x (cities, 30)
        return {"rain": np.stack(members, axis=1).tolist()}


runs_rf = evaluate(Forest(forests))
print(runs_rf.mean().round(4).to_string())
print("forest - climatology: %+.4f ± %.4f" % compare(runs_rf, runs_clim))

CRPS 6h     0.1020
CRPS 48h    0.1132
score       0.1076
forest - climatology: +0.0317 ± 0.0006


**Finding.** The forest scores 0.1076 (0.1020 at +6 h, 0.1132 at
+48 h): 0.0317 ± 0.0006 worse than the climatology, and worse than
forecasting 0 mm everywhere (0.0804, the analysis notebook). It reads
the weather; the climatology reads nothing.

**Question.** Why? Each tree predicts the mean of its leaf. Compare
the forest's members with the climatology's for one city and one run
— how many are 0, how far apart they are — and say which term of the
CRPS the forest loses on.

*Your answer:*

---

## 6. Your turn

Build features, fit a model, and score it here before you submit it.

- **Features.** Anything computed from the request. The analysis
  notebook lists the candidates: rain in the city upwind (section 7),
  the solar hour in summer (section 5), a 6-hour change with the time
  of day beside it (section 8). Write them in `features`: `window(i)`
  gives it in training exactly what the request gives it live.
- **Model.** Anything whose `predict(request)` returns (cities, 2, M)
  members, 1 ≤ M ≤ 100. The members are a distribution: a spike at
  0 and a long tail. A hurdle model (a classifier for P(wet), a
  distribution of amounts given wet, its quantiles as members) or one
  quantile regressor per level both produce one.
- **Evaluate.** `evaluate(agent, issues)`, then `compare` with the
  climatology on the same runs. A gap is real when it is several
  times its ±.

The bars, over the same replay: the climatology, 0.0759; the
challenge's starter agent (the template on the challenge page),
0.0746; a +6 h climatology split by rain in the last 3 hours, with
this climatology at +48 h, 0.0742.

Every look at the replay is a look at the test period. Tune on
2024, fitted on 2020–2023 (the cell below), and replay 2025–2026 once
per model you would submit, refitted on 2020–2024. To submit, write
`agent.py` as in the agent notebook, save the models next to it
(`joblib.dump`), and submit with `runtime_id=182`, the runtime with
scikit-learn, LightGBM and XGBoost.

In [26]:
VALID = hours.searchsorted(pd.Timestamp("2024-01-01", tz="UTC"))
FIT, VAL = issue_times(0, VALID), issue_times(VALID, SPLIT)
clim_val = evaluate(Climatology(VALID), VAL)
print(clim_val.mean().round(4).to_string())

# Your model: fit on FIT only, then
# mine = ...                          # anything with .predict(request)
# runs_mine = evaluate(mine, VAL)
# print("mine - climatology: %+.4f ± %.4f" % compare(runs_mine, clim_val))

CRPS 6h     0.0811
CRPS 48h    0.0808
score       0.0809


### 6.1 Features and modelling choices

Features use only the request's **48-hour history, issue time, and fixed city coordinates**. The 51 candidates include recent rain and wet-hour frequency; 3/6/24-hour weather summaries and 6-hour changes; wind components; nearby and upwind weather; coordinates; cyclic season and local solar hour; and humidity interactions. Visibility is excluded because it is entirely missing. Upwind weights favour cities in the direction **from which** the local wind blows. Fixed coordinates come from the request's city panel.

Each horizon has a **hurdle distribution**, rather than an ensemble of point estimates:

1. Estimate the probability of rain.
2. Regress **log rain amount on wet observations only**.
3. Use held-out probability calibration and empirical log-amount residual quantiles to produce **100 equally weighted members**, including exact zeros.

**Linear regression** uses Ridge for both a linear probability model and log amounts. **SVM** uses a support vector classifier and regressor with a reproducible, 256-component approximation to an RBF kernel, keeping training practical on this dataset. **XGBoost** uses boosted classification and regression trees. The residual distribution is pooled within each horizon; this is a compact approximation to conditional uncertainty. Direct XGBoost quantile regression was also explored on 2024; it did not beat the hurdle approach.

For every candidate, median imputation, scaling, feature selection, and model fitting use only the **core training years**. The final training year is reserved for isotonic probability calibration and amount residuals: **2020–2022 / 2023** when validating on **2024**, then **2020–2023 / 2024** for the final replay. Training targets stop before the calibration boundary, and all fitting/calibration targets stop before the evaluation boundary. The original `window` already replaces missing feed values with zero; the imputer additionally handles any non-finite engineered values.

We compare all 51 features with the top 24 by training-only wet/dry F statistics. We tune Ridge regularisation, SVM kernel width, and XGBoost **(depth, number of trees)**. For XGBoost, scaling is unnecessary but harmless; the shared preprocessing keeps the code small. The grid and model family are selected by **2024 CRPS**, never by replay results. The ± values use the notebook's weekly-block standard errors.

Requires XGBoost (`%pip install -q xgboost` if absent). The rest of the notebook is unchanged. Added code: under 200 lines, including tuning and the final replay.


In [28]:
from itertools import product
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier, XGBRegressor

lat, lon = coords.to_numpy().T
north = (lat[None, :] - lat[:, None]) * 111
east = (lon[None, :] - lon[:, None]) * 111 * np.cos(np.deg2rad(lat[:, None]))
distance = np.hypot(north, east)
nearby = np.exp(-distance / 300) * (distance > 0)
nearby /= nearby.sum(axis=1, keepdims=True)

def rain_features(history, issue):
    t, r, w, d, humid, cloud, _, snow = np.asarray(history).transpose(2, 0, 1)
    angle = np.deg2rad(d[:, -1])
    alignment = (east * np.sin(angle[:, None]) + north * np.cos(angle[:, None]))
    alignment /= np.maximum(distance, 1)
    upwind = np.exp(-distance / 500) * np.maximum(alignment, 0) ** 4
    upwind *= distance > 0
    upwind /= np.maximum(upwind.sum(axis=1, keepdims=True), 1e-8)
    season = 2 * np.pi * issue.dayofyear / 365.25
    solar = 2 * np.pi * (issue.hour + lon / 15) / 24
    columns = {}
    for name, values in zip(['temperature', 'rain', 'wind', 'humidity', 'clouds', 'snow'],
                            [t, r, w, humid, cloud, snow]):
        columns[name] = values[:, -1]
        for k in [3, 6, 24]:
            columns[f'{name}_mean{k}'] = values[:, -k:].mean(axis=1)
        columns[f'{name}_change6'] = values[:, -1] - values[:, -7]
    columns.update(rain_max6=r[:, -6:].max(axis=1), rain_wet24=(r[:, -24:] > 0).mean(axis=1),
                   east_wind=w[:, -1] * np.sin(angle), north_wind=w[:, -1] * np.cos(angle))
    for name, values in [('rain', r[:, -3:].mean(axis=1)), ('humidity', humid[:, -1]),
                         ('clouds', cloud[:, -1]), ('temperature', t[:, -1])]:
        columns['upwind_' + name] = upwind @ values
        columns['nearby_' + name] = nearby @ values
    columns.update(latitude=lat, longitude=lon, season_sin=np.sin(season),
                   season_cos=np.cos(season), solar_sin=np.sin(solar), solar_cos=np.cos(solar),
                   summer_solar=np.cos(season) * np.cos(solar),
                   humid_cloud=humid[:, -1] * cloud[:, -1] / 100,
                   humid_temp=humid[:, -1] * t[:, -1] / 100)
    return pd.DataFrame(columns).replace([np.inf, -np.inf], np.nan)

def design(issues):
    return pd.concat([rain_features(window(i), hours[i]) for i in issues], ignore_index=True)



### 6.2 Fit the occurrence and positive-amount models


In [29]:
def risk(model, Z):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(Z)[:, 1]
    return model.decision_function(Z) if hasattr(model, 'decision_function') else model.predict(Z)

class RainModel:
    def __init__(self, family, k, parameter):
        self.family, self.k, self.parameter = family, k, parameter
        self.models, self.selected = {}, {}

    def fit(self, F, issues, end):
        year = hours[end - 1].year
        calibration_start = hours.searchsorted(pd.Timestamp(f'{year}-01-01', tz='UTC'))
        core = np.repeat(issues + max(HORIZONS) < calibration_start, len(PANEL))
        cal = np.repeat(issues >= calibration_start, len(PANEL))
        for h in HORIZONS:
            y = rain[issues + h].reshape(-1)
            wet = y > 0
            steps = [SimpleImputer(strategy='median'), StandardScaler(),
                     SelectKBest(f_classif, k=self.k)]
            if self.family == 'SVM':
                steps += [RBFSampler(gamma=self.parameter, n_components=256, random_state=0)]
            pre = make_pipeline(*steps).fit(F[core], wet[core])
            self.selected[h] = F.columns[pre.named_steps['selectkbest'].get_support()].tolist()
            A, B = pre.transform(F[core]), pre.transform(F[cal])
            if self.family == 'Linear regression':
                occurrence, amount = Ridge(alpha=self.parameter), Ridge(alpha=self.parameter)
            elif self.family == 'SVM':
                occurrence = LinearSVC(C=.1, dual=False, max_iter=5000)
                amount = LinearSVR(C=.1, epsilon=.1, dual=False,
                                   loss='squared_epsilon_insensitive', max_iter=5000)
            else:
                depth, trees = self.parameter
                settings = dict(n_estimators=trees, max_depth=depth, learning_rate=.05,
                                min_child_weight=100, subsample=.8, colsample_bytree=.9,
                                reg_lambda=20, n_jobs=6, random_state=0, tree_method='hist')
                occurrence, amount = XGBClassifier(**settings), XGBRegressor(**settings)
            occurrence.fit(A, wet[core])
            amount.fit(A[wet[core]], np.log(y[core & wet]))
            calibrator = IsotonicRegression(out_of_bounds='clip').fit(risk(occurrence, B), wet[cal])
            residual = np.log(y[cal & wet]) - amount.predict(B[wet[cal]])
            self.models[h] = pre, occurrence, amount, calibrator, np.quantile(residual, LEVELS)
        return self

    def members(self, F, horizons=HORIZONS):
        ensembles = []
        for h in horizons:
            pre, occurrence, amount, calibrator, residuals = self.models[h]
            Z = pre.transform(F)
            p = np.clip(calibrator.predict(risk(occurrence, Z)), .001, .999)
            q = (LEVELS[None, :] - (1 - p[:, None])) / p[:, None]
            residual = np.interp(np.maximum(q, 0), LEVELS, residuals)
            log_amount = amount.predict(Z)[:, None] + residual
            ensembles.append(np.where(q > 0, np.exp(np.clip(log_amount, -5, 5)), 0))
        return np.stack(ensembles, axis=1)

    def predict(self, request):
        F = rain_features(request['history'], pd.Timestamp(request['issue_time']))
        return {'rain': self.members(F, request['horizons']).tolist()}



### 6.3 Choose features and parameters on 2024


In [30]:
F_fit, F_val = design(FIT), design(VAL)
Y_val = np.stack([rain[VAL + h] for h in HORIZONS], axis=-1).reshape(-1, 2)
grid = {'Linear regression': [1, 1000], 'SVM': [.003, .01],
        'XGBoost': [(3, 250), (5, 250), (7, 250), (5, 500), (7, 500)]}
records, best = [], {}
for family, parameters in grid.items():
    for k, parameter in product(['all', 24], parameters):
        model = RainModel(family, k, parameter).fit(F_fit, FIT, VALID)
        loss = crps(model.members(F_val), Y_val).reshape(len(VAL), len(PANEL), 2).mean(axis=1)
        runs = pd.DataFrame(loss, columns=['CRPS 6h', 'CRPS 48h'], index=hours[VAL])
        runs['score'] = runs.mean(axis=1)
        delta, se = compare(runs, clim_val)
        row = dict(model=family, k=k, parameter=parameter, **runs.mean().to_dict(), delta=delta, SE=se)
        records.append(row)
        if family not in best or row['score'] < best[family][0]['score']:
            best[family] = row, model
tuning = pd.DataFrame(records).sort_values('score')
print(tuning.round(6).to_string(index=False))
winner = min(best, key=lambda family: best[family][0]['score'])
print('\nSelected on 2024:', winner)



            model   k parameter  CRPS 6h  CRPS 48h    score     delta       SE
          XGBoost all  (7, 500) 0.072461  0.080016 0.076239 -0.004706 0.000386
          XGBoost all  (5, 500) 0.072581  0.079897 0.076239 -0.004705 0.000376
          XGBoost all  (7, 250) 0.072559  0.079928 0.076244 -0.004701 0.000384
          XGBoost all  (5, 250) 0.072804  0.079866 0.076335 -0.004609 0.000371
          XGBoost all  (3, 250) 0.073225  0.079890 0.076558 -0.004386 0.000367
          XGBoost  24  (7, 250) 0.073710  0.080058 0.076884 -0.004060 0.000370
          XGBoost  24  (5, 250) 0.073759  0.080029 0.076894 -0.004050 0.000361
          XGBoost  24  (5, 500) 0.073750  0.080059 0.076904 -0.004040 0.000362
          XGBoost  24  (7, 500) 0.073789  0.080138 0.076963 -0.003981 0.000363
          XGBoost  24  (3, 250) 0.073976  0.080024 0.077000 -0.003944 0.000350
              SVM all     0.003 0.074030  0.080125 0.077078 -0.003866 0.000356
              SVM all      0.01 0.074303  0.080159 0

### 6.4 Verify predictions through the original request/replay API


In [31]:
for family, (row, model) in best.items():
    batch = model.members(F_val.iloc[:len(PANEL)])
    assert np.allclose(check(model.predict(make_request(VAL[0])), make_request(VAL[0])), batch)
    runs = evaluate(model, VAL)
    assert np.isclose(runs['score'].mean(), row['score'])
    print(family, 'validation - climatology: %+.6f ± %.6f' % compare(runs, clim_val))
    print('Selected features:', {h: len(names) for h, names in model.selected.items()})



Linear regression validation - climatology: -0.003354 ± 0.000309
Selected features: {6: 51, 48: 51}
SVM validation - climatology: -0.003866 ± 0.000356
Selected features: {6: 51, 48: 51}
XGBoost validation - climatology: -0.004706 ± 0.000386
Selected features: {6: 51, 48: 51}


### 6.5 Refit and assess once on 2025–2026


In [32]:
F_train = design(TRAIN)
final_models = {family: RainModel(family, row['k'], row['parameter']).fit(F_train, TRAIN, SPLIT)
                for family, (row, _) in best.items()}
replay, summary = {}, []
for family, model in final_models.items():
    replay[family] = evaluate(model, TEST)
    delta, se = compare(replay[family], runs_clim)
    summary.append(dict(model=family, **replay[family].mean().to_dict(), delta=delta, SE=se))
print(pd.DataFrame(summary).set_index('model').round(6).to_string())
agent = final_models[winner]
runs_mine = replay[winner]
print('\nFinal model (chosen on 2024):', winner)
print('Improvement over climatology: %.2f%%' % (100 * (1 - runs_mine['score'].mean() / runs_clim['score'].mean())))


                    CRPS 6h  CRPS 48h     score     delta        SE
model                                                              
Linear regression  0.069586  0.075071  0.072328 -0.003566  0.000257
SVM                0.068897  0.074876  0.071887 -0.004008  0.000273
XGBoost            0.067438  0.074747  0.071093 -0.004802  0.000327

Final model (chosen on 2024): XGBoost
Improvement over climatology: 6.33%


**Finding.** The model chosen on **2024** is **XGBoost**, with all 51 features, depth 7 and 500 trees in each stage. On the single **2025–2026** replay, its CRPS is **0.071093**, versus **0.075895** for climatology: **6.33% lower**. The paired difference is **-0.004802 ± 0.000327** (weekly-block standard error). The reduction is 14.7 times its standard error. Neither model choice nor parameters were changed after this replay.

All 24-feature variants were weaker on validation, so the full feature set is retained. `agent` holds the selected, refitted model and `runs_mine` its replay results.


### 6.6 Submit the selected model

Run this cell **after section 6.5**. It packages `final_models[winner]`, the XGBoost model selected on 2024, with its feature extraction, preprocessing, and calibration. The saved agent is checked against the original model before **uploading and deploying** it to challenge 177 with **runtime 182**, using the existing API key.

The two files are written to `rain_submission/`. XGBoost trees use native JSON inside the bundle; the exported agent needs no notebook classes or training data. The package has also been tested with the runtime's XGBoost 3.2.0, with identical predictions. **Submission 9439 (`rain-xgboost-best`) was deployed from these files on 19 October 2026 and is active on runtime 182.** Running the cell again creates another submission.


In [ ]:
import inspect
import importlib.util
from pathlib import Path
import joblib
submission_dir = Path('rain_submission')
submission_dir.mkdir(exist_ok=True)
best_agent = final_models[winner]  # Winner was selected on 2024, before the replay.
assert best_agent.family == 'XGBoost'
models = {h: (pre, wet.get_booster().save_raw(raw_format='json'),
              amount.get_booster().save_raw(raw_format='json'), cal, residuals)
          for h, (pre, wet, amount, cal, residuals) in best_agent.models.items()}
joblib.dump((models, LEVELS, HORIZONS, lat, lon, north, east, distance, nearby),
            submission_dir / 'model.joblib', compress=3)
header = '''from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
(models, LEVELS, HORIZONS, lat, lon, north, east, distance, nearby) = joblib.load(
    Path(__file__).with_name('model.joblib'))
'''
constructor = '''class Agent:
    def __init__(self):
        self.models = {}
        for h, (pre, wet_json, amount_json, cal, residuals) in models.items():
            wet, amount = XGBClassifier(), XGBRegressor()
            wet.load_model(wet_json)
            amount.load_model(amount_json)
            self.models[h] = pre, wet, amount, cal, residuals
'''
source = '\n\n'.join([header, inspect.getsource(rain_features), inspect.getsource(risk),
                       constructor + inspect.getsource(RainModel.members) + inspect.getsource(RainModel.predict)])
(submission_dir / 'agent.py').write_text(source)
spec = importlib.util.spec_from_file_location('submission_agent', submission_dir / 'agent.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
request = make_request(VAL[0])
np.testing.assert_allclose(check(module.Agent().predict(request), request),
                           check(best_agent.predict(request), request), rtol=1e-6, atol=1e-8)
client = mlarena.connect(os.environ.get('MLARENA_API_KEY') or API_KEY)
submission_id = client.create_submission(CHALLENGE_ID, 'rain-xgboost-best')['submission_id']
client.set_agent_runtime(submission_id, 182)
for name in ['agent.py', 'model.joblib']:
    client.upload_submission_file(CHALLENGE_ID, submission_id, str(submission_dir / name))
submission = client.deploy_submission(CHALLENGE_ID, submission_id)
print({'submission_id': submission_id, **submission})
